# 04 · LLM: concordância com humanos e rotulagem fraca

1. Mede a concordância do LLM com o gabarito humano na **validação** e calibra o prompt.
2. Avalia o LLM direto no **teste** (modelo de referência).
3. Rotula o **treino do RePro** (mesmos textos dos rótulos humanos) para comparar fonte de rótulo humano × LLM.
4. Rotula comentários da **Olist** para treino no domínio de aplicação.

Ambiente: **GPU T4**. Tudo fica em cache: se a sessão cair, rode de novo que continua de onde parou.

In [ ]:
# Configuração: no Colab, clona o repositório, instala o pacote `absa`
# e guarda cache do LLM, modelos e predições no Drive (sobrevive à queda da sessão)
import sys
if "google.colab" in sys.modules:
    !git clone -q https://github.com/vnevesl/pln-absa-olist.git 2>/dev/null || git -C pln-absa-olist pull -q
    %cd pln-absa-olist
    !pip -q install -e ".[llm]"
    from google.colab import drive
    drive.mount("/content/drive")
    !mkdir -p /content/drive/MyDrive/pln-absa-olist/llm_cache /content/drive/MyDrive/pln-absa-olist/models /content/drive/MyDrive/pln-absa-olist/interim
    !ln -sfn /content/drive/MyDrive/pln-absa-olist/llm_cache data/llm_cache
    !ln -sfn /content/drive/MyDrive/pln-absa-olist/models models
    !ln -sfn /content/drive/MyDrive/pln-absa-olist/interim data/interim

In [ ]:
import pandas as pd
from absa.config import ASPECTS, INTERIM, LABEL_COLUMNS, SEED
from absa.llm_labeler import hf_generator, label_texts
from absa.metrics import bootstrap_ci, per_aspect_report, tuple_f1
from absa.preprocess import clean_light
from absa.repro import load_repro
treino, val, teste, _ = load_repro()
print(len(treino), len(val), len(teste))

# Few-shot: exemplos fixos da validação (ficam fora da avaliação)
amostra = val.sample(6, random_state=SEED)
exemplos = [{"texto": clean_light(r.texto), **{a: r[f"asp_{a}"] for a in ASPECTS}} for _, r in amostra.iterrows()]
val_aval = val.drop(amostra.index)

In [ ]:
MODELO = "Qwen/Qwen2.5-7B-Instruct"
generate = hf_generator(MODELO)  # alternativa: openai_compatible_generator()

In [ ]:
def rotular(df, cache):
    rot, _ = label_texts(df.texto.map(clean_light).tolist(), generate, exemplos, cache_name=cache)
    print(f"\nrespostas válidas: {sum(r is not None for r in rot)}/{len(rot)}")
    # Resposta inválida vira "ausente" em tudo (conta como erro na avaliação)
    return pd.DataFrame([{f"asp_{a}": (r or {}).get(a, "ausente") for a in ASPECTS} for r in rot])

## 1) Concordância na validação

Se estiver baixa, ajuste o prompt em `src/absa/llm_labeler.py` (ou os exemplos) e repita esta etapa **antes** de seguir. Nunca ajuste olhando o teste.

In [ ]:
pred_val = rotular(val_aval, "qwen7b_repro_val")
print("F1 tupla:", round(tuple_f1(val_aval, pred_val), 3))
per_aspect_report(val_aval, pred_val).round(3)

## 2) LLM direto no teste

In [ ]:
pred_teste = rotular(teste, "qwen7b_repro_teste")
pred_teste.assign(review_id=teste.review_id.values).to_csv(INTERIM / "pred_repro_teste_llm_fewshot.csv", index=False)
f1, lo, hi = bootstrap_ci(teste, pred_teste)
print(f"LLM few-shot: F1 macro {f1:.3f} (IC95% {lo:.3f}–{hi:.3f})")

## 3) Rótulos do LLM para o treino do RePro

In [ ]:
pred_treino = rotular(treino, "qwen7b_repro_treino")
treino[["review_id", "texto", "nota"]].join(pred_treino).to_csv(INTERIM / "repro_treino_rotulos_llm.csv", index=False)
# Concordância LLM × humano no treino (quanto ruído os rótulos fracos carregam)
print("F1 tupla LLM vs humano no treino:", round(tuple_f1(treino, pred_treino), 3))

## 4) Rótulos do LLM para a Olist

In [ ]:
from absa.data import load_reviews
olist = load_reviews()
pool = olist[olist.n_palavras >= 3].drop_duplicates("texto")
amostra_olist = pd.concat(g.sample(min(len(g), 1200), random_state=SEED) for _, g in pool.groupby("nota"))
pred_olist = rotular(amostra_olist, "qwen7b_olist")
amostra_olist[["review_id", "texto", "nota", "atrasado"]].reset_index(drop=True).join(pred_olist) \
    .to_csv(INTERIM / "olist_rotulos_llm.csv", index=False)
pred_olist.apply(lambda s: s.value_counts()).fillna(0).astype(int)